In [11]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

true_df = pd.read_csv(path + '/True.csv')
fake_df = pd.read_csv(path + '/Fake.csv')

print("Path to dataset files:", path)

Path to dataset files: /home/jens/.cache/kagglehub/datasets/clmentbisaillon/fake-and-real-news-dataset/versions/1


In [12]:
true_df.describe()

,title,text,subject,date
count,21417,21417,21417,21417
unique,20826,21192,2,716
top,Factbox: Trump fills top jobs for his administ...,(Reuters) - Highlights for U.S. President Dona...,politicsNews,"December 20, 2017"
freq,14,8,11272,182


In [13]:
true_df.sample(5, random_state=42)

,title,text,subject,date
18137,Europe rights watchdog says Turkey's emergency...,BRUSSELS (Reuters) - A leading European rights...,worldnews,"October 6, 2017"
3277,Exclusive: Trump targets illegal immigrants wh...,"(Reuters) - In September 2014, Gilberto Velasq...",politicsNews,"June 9, 2017"
2876,"At G20 summit, Trump pledges $639 million in a...",HAMBURG (Reuters) - U.S. President Donald Trum...,politicsNews,"July 8, 2017"
5160,Ex-Christie associates lose bid for new trial ...,NEW YORK (Reuters) - A federal judge rejected ...,politicsNews,"March 2, 2017"
10843,Young blacks more open to Bernie Sanders' Whit...,"ORANGEBURG, S.C. (Reuters) - If Democratic hop...",politicsNews,"February 12, 2016"


In [14]:
fake_df.sample(5, random_state=42)

,title,text,subject,date
13474,ABOUT HILLARY’S COUGH: We Discovered The Secre...,,politics,"Jul 20, 2016"
11994,BREAKING: OBAMACARE REPEAL Clears First Hurdle...,The Senate voted 51-48 this afternoon to proce...,politics,"Jan 4, 2017"
19179,‘SLEEPY’ JUSTICE GINSBURG: Excites Crowd By Sa...,So much for the SCOTUS not being political Che...,left-news,"Feb 7, 2017"
501,WATCH: Kellyanne Conway Very Upset Hillary Cl...,White House counselor Kellyanne Conway crawled...,News,"August 24, 2017"
3492,"GOP Gives Trump The Middle Finger, Prepares T...",Donald Trump may have decided that Russia is g...,News,"December 9, 2016"


In [15]:
# Combine datasets
fake_df['is_fake'] = 1
true_df['is_fake'] = 0

df = pd.concat([fake_df, true_df])

df.sample(5, random_state=42)


,title,text,subject,date,is_fake
22216,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1
4436,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0
1526,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0
1377,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",1
8995,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",0


In [16]:
# Clean dataset by dropping columns and duplicates
print(df.shape)
clean_df = df.copy().drop(columns = ['title','subject','date'])
clean_df.drop_duplicates(inplace=True)
print(clean_df.shape)
clean_df.sample(5, random_state=42)

(44898, 5)
(38647, 2)


,text,is_fake
18652,(This September 29 has been corrected to fix ...,0
5890,(Reuters) - U.S. President Donald Trump said o...,0
21039,PRISTINA (Reuters) - Kosovo s center-right coa...,0
4137,WASHINGTON (Reuters) - President Donald Trump ...,0
4156,A Pennsylvania man apparently failed basic bio...,1


In [34]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    sanitized_text = re.sub(r'[^a-zA-Z\s]', '', text)
    sanitized_text = sanitized_text.lower()
    split_text = sanitized_text.split()
    text_without_stopwords = [word for word in split_text if word not in stop_words]
    lemmatized_text = [lemmatizer.lemmatize(word) for word in text_without_stopwords]
    joined_tokens = ' '.join(lemmatized_text)
    return joined_tokens

clean_df['text'] = clean_df['text'].apply(preprocess_text)
clean_df.sample(5, random_state=42)

[nltk_data] Downloading package stopwords to /home/jens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jens/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,text,is_fake
18652,september corrected fix date election paragrap...,0
5890,reuters u president donald trump said friday a...,0
21039,pristina reuters kosovo centerright coalition ...,0
4137,washington reuters president donald trump frid...,0
4156,pennsylvania man apparently failed basic biolo...,1


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(clean_df['text'])

print("Shape of TF-IDF matrix:", tfidf_matrix.shape)

# x = clean_df['text'].values
# y = clean_df['is_fake'].values
#
# model = LogisticRegression()
# model.fit(X_train, y_train)

Shape of TF-IDF matrix: (38647, 203321)
